# Building the Cu–Fe–S–As "4D" Scatter Plot with Plotly
### Mungana & Red Dome Cu Mineralisation Diagram

This notebook walks through **exactly how the original script builds the diagram**, step by step, with explanations of *why* each step is done and what it means geologically/statistically.

The chart encodes **four variables at once**:

| Visual channel | Variable | Meaning |
|---|---|---|
| X axis | `S_pct` | Sulphur content (%) |
| Y axis | `Fe_pct` | Iron content (%) |
| Marker **size** | `Cu_pct` | Copper content (%) — bigger = higher grade |
| Marker **color** | `As_ppm` | Arsenic content (ppm) — a penalty/deleterious element |
| Marker **symbol** | `As_Group` | Square = As ≤ 2000 ppm, Triangle = As > 2000 ppm (a redundant but very readable encoding of the same As information) |

A dashed red line (the **pyrite tie-line**, after Escolme et al., 2017) is overlaid as a petrological reference: points lying near/on this line are consistent with a simple pyrite (FeS₂) signature, while points sitting well off the line suggest a different Fe–S mineral assemblage (e.g. chalcopyrite, pyrrhotite, or oxide/gossanous material where S has been lost).

> **Note on data**: the original script reads three site-specific CSV files (`845.csv`, `883.csv`, `997.csv`) from a local Windows path. Since those files aren't available here, this notebook generates a **synthetic but realistic Cu–Fe–S–As dataset** so every cell below is fully runnable end-to-end. The logic, column names, and plotting code are otherwise a faithful, cleaned-up reproduction of the original script — swap in your real CSV paths in Section 1 and everything downstream works unchanged.


## 0. Imports

Only three libraries are needed:
- `pandas` for tabular data loading/cleaning
- `plotly.express` for the interactive scatter plot
- `numpy` for the vectorised `As_Group` classification


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px


## 1. Load and merge the assay data

The original workflow assumes assay results are split across several per-drillhole-batch CSV files (`845.csv`, `883.csv`, `997.csv` — likely drillhole or sample-batch IDs from the Mungana/Red Dome project). The pattern is:

1. Read each CSV into its own DataFrame.
2. Concatenate them into a single DataFrame (`pd.concat`).
3. Save that merged table straight back to disk **before any cleaning**.

That last step matters for reproducibility/auditability in a resource-estimation context — you keep an untouched "as-merged" copy of the raw data so any later cleaning decision can be traced back and re-checked, and so you don't have to re-read three separate files every time.

```python
df_845 = pd.read_csv(r"...\845.csv")
df_883 = pd.read_csv(r"...\883.csv")
df_997 = pd.read_csv(r"...\997.csv")

df = pd.concat([df_845, df_883, df_997])
df.to_csv(r"...\merged.csv")
```

Below we simulate the same shape of raw data — three batches with the same columns, some missing/non-numeric values thrown in (as real assay data often has, e.g. `<0.01`, blanks, or `n/a` for below-detection results) — and merge them the same way.


In [ ]:
rng = np.random.default_rng(42)

def make_batch(n, batch_id):
    """Simulate one assay batch, mimicking real lab-result quirks
    (a few missing values, a few below-detection-limit strings)."""
    fe_pct = rng.gamma(3.0, 4.0, n).round(2)                     # 0-40% ish, right-skewed
    s_pct  = np.clip(fe_pct/2 + rng.normal(0, 2.5, n), 0, None).round(2)
    cu_ppm = (rng.lognormal(mean=8.5, sigma=1.1, size=n)).round(0)  # ppm -> up to a few %
    as_ppm = (rng.lognormal(mean=6.0, sigma=1.6, size=n)).round(0)

    batch = pd.DataFrame({
        'Hole_ID': [f'{batch_id}-{i:03d}' for i in range(n)],
        'Fe_pct': fe_pct,
        'S_pct': s_pct,
        'Cu_ppm': cu_ppm,
        'As_ppm': as_ppm,
    })

    # simulate a handful of missing / below-detection entries, like real lab sheets
    batch['As_ppm'] = batch['As_ppm'].astype(object)
    n_bad = max(1, n // 25)
    bad_idx = rng.choice(batch.index, size=n_bad, replace=False)
    batch.loc[bad_idx, 'As_ppm'] = rng.choice(['<5', '', 'n/a'], size=n_bad)

    return batch

df_845 = make_batch(70, 845)
df_883 = make_batch(55, 883)
df_997 = make_batch(40, 997)

df_list = [df_845, df_883, df_997]
df = pd.concat(df_list, ignore_index=True)

df_cp = df.copy()

# In the original script this is written to merged.csv for auditability:
# df_cp.to_csv(r"...\merged.csv")
df_cp.to_csv("merged.csv", index=False)

df_cp.head()


## 2. Re-read the merged file and select the relevant elements

The script deliberately **re-reads the CSV it just wrote** (`df_merged = pd.read_csv(...)`) rather than continuing to use `df_cp` in memory. This is a common defensive pattern in geoscience/data-QA workflows: it guarantees the DataFrame you clean is *exactly* what's on disk (catching any silent dtype/encoding surprises introduced by the write/read round-trip) and keeps the "merge" and "clean" stages fully decoupled.

Then it narrows the table down to the four elements that matter for this interpretation:

- **Cu** (`Cu_ppm`) — the target/ore metal
- **Fe** (`Fe_pct`) and **S** (`S_pct`) — the two elements that define chalcopyrite (CuFeS₂) and pyrite (FeS₂) stoichiometry
- **As** (`As_ppm`) — arsenic, a classic **penalty element**: smelters heavily discount or reject concentrate with high arsenic, so it's tracked even though it's not an ore metal.


In [ ]:
df_merged = pd.read_csv("merged.csv")
df_merged_cp = df_merged.copy()

cols = ['Cu_ppm', 'Fe_pct', 'S_pct', 'As_ppm']
df_merged_cp.head()


## 3. Force the assay columns to numeric and drop unusable rows

Assay spreadsheets routinely contain non-numeric placeholders for below-detection-limit results (`<5`, `n/a`, blanks, etc.). `pd.to_numeric(..., errors='coerce')` converts every genuinely numeric string to a float and turns anything it can't parse into `NaN`. `dropna(subset=cols)` then removes any row that isn't fully numeric across all four elements — i.e. a sample is only kept if it has a valid Cu, Fe, S, **and** As result.

This is a conservative but standard choice: rather than guessing a substitute value for missing/below-detection data (which would bias the size/color encoding), incomplete rows are simply excluded from this particular 4-element plot.


In [ ]:
df_merged_cp[cols] = df_merged_cp[cols].apply(pd.to_numeric, errors='coerce')
df_clean = df_merged_cp.dropna(subset=cols)

print(f"Rows before cleaning: {len(df_merged_cp)}")
print(f"Rows after cleaning:  {len(df_clean)}")
df_clean[cols].describe()


## 4. Convert Cu to percent, and build the working table

`Cu_ppm` is divided by 10,000 to convert to `Cu_pct` (1% = 10,000 ppm), because a percentage is a far more intuitive size scale for the bubble chart than a raw ppm value, and it matches the units geologists usually think in for Cu grade (Fe and S are already in %).

The resulting table — `CuFeS_mineral` — is the fully clean, analysis-ready dataset, and is saved to its own CSV so downstream plotting scripts don't need to repeat the cleaning steps.


In [ ]:
CuFeS_mineral = df_clean.loc[:, cols].copy()
CuFeS_mineral['Cu_pct'] = CuFeS_mineral['Cu_ppm'] / 10_000

# Saved so any later notebook/script can start from here directly:
CuFeS_mineral.to_csv("merged_Cu_pct.csv", index=False)

CuFeS_mineral.head()


## 5. Filter to economic-grade copper and flag the arsenic groups

Two domain-driven decisions happen here:

1. **`Cu_pct > 1.0`** — 1% Cu is used as a typical rule-of-thumb economic cut-off grade for a Cu deposit. Filtering to this subset focuses the plot on material that's actually of *ore-grade interest*, rather than diluting the picture with low-grade/waste samples.
2. **`As_Group`** — `np.where` is used to vectorise a simple binary classification: samples with more than 2000 ppm As are labelled `"Above 2000"`, everything else `"Below 2000"`. This becomes the marker **shape** (triangle vs square), giving a second, easy-to-scan visual cue for arsenic risk on top of the continuous color scale.

Why 2000 ppm specifically for the *shape* threshold, while the *color* scale later runs up to 15,000 ppm? Because 2000 ppm is being used as a practical "watch-list" trigger for processing risk, while the comment in the script also notes 5000 ppm as a general safety/penalty ceiling for arsenic in concentrate — the color scale is deliberately set wider (0–15,000) so that even the most extreme outliers are still visible as shading rather than all clipping to the same maximum color.


In [ ]:
CuFeS_mineral_filtered = CuFeS_mineral[CuFeS_mineral['Cu_pct'] > 1.0].copy()

CuFeS_mineral_filtered["As_Group"] = np.where(
    CuFeS_mineral_filtered["As_ppm"] > 2000, "Above 2000", "Below 2000"
)

print(f"Samples above 1% Cu: {len(CuFeS_mineral_filtered)} of {len(CuFeS_mineral)}")
CuFeS_mineral_filtered["As_Group"].value_counts()


## 6. Build the 4-variable scatter plot

`px.scatter` does the heavy lifting, mapping:

| Parameter | Column | Effect |
|---|---|---|
| `x` | `S_pct` | horizontal position |
| `y` | `Fe_pct` | vertical position |
| `size` | `Cu_pct` | bubble size ∝ Cu grade |
| `color` | `As_ppm` | continuous color scale (Viridis) |
| `symbol` | `As_Group` | square vs triangle-up marker shape |
| `symbol_map` | — | explicitly pins which shape goes with which group, so it's stable even if group order changes |
| `range_x` / `range_y` | — | fixes the axis window so multiple plots (e.g. before/after filtering, or different As thresholds) stay visually comparable |
| `range_color` | `[0, 15000]` | fixes the color scale so color always means the same As concentration across different runs/subsets of the plot |

Using **Viridis** for the continuous color scale is a good practice choice here: it's perceptually uniform and colorblind-friendly, so the color gradient reads consistently regardless of who's viewing it.


In [ ]:
diagram = px.scatter(
    CuFeS_mineral_filtered,
    x='S_pct',
    y='Fe_pct',
    size='Cu_pct',
    color='As_ppm',
    symbol="As_Group",
    symbol_map={"Above 2000": "triangle-up", "Below 2000": "square"},
    range_x=[0, 20],
    range_y=[0, 35],
    range_color=[0, 15000],
    color_continuous_scale=px.colors.sequential.Viridis,
    title="Mungana and Red Dome Cu mineralisation. Color = As content (ppm), "
          "Size = Cu content (%). Square = low As, triangle = high As"
)

diagram.show()


## 7. Overlay the pyrite tie-line

This is a **reference line**, not data — it's added with `add_shape`, not another `scatter` trace. The line is defined by two Fe–S points taken from Escolme et al. (2017) that characterise the ideal pyrite (FeS₂) stoichiometric ratio on an S(x)–Fe(y) plot:

- Point 0: `(S=3, Fe=2.5)`
- Point 1: `(S=6, Fe=5)`

From these two points, the script computes the line's **slope** and **intercept** algebraically (`slope = Δy/Δx`, `intercept = y0 - slope·x0`), then **extrapolates** that line across the full x-range of the plot (`x_range = [0, 15]`) so it spans the whole chart, not just the segment between the two defining points.

Geologically: samples sitting close to this dashed line have an Fe:S ratio consistent with (nearly) pure pyrite. Samples well **above** the line are Fe-rich relative to S (e.g. magnetite/hematite-bearing, or Fe silicates), and samples well **below** the line are S-rich relative to Fe (possibly reflecting other sulphides like chalcopyrite or excess S). It's a quick visual sanity check for "how much of this Fe–S signature is just pyrite?"

> ⚠️ The comment in the source script flags this explicitly: **the tie-line is only valid when S is on the x-axis and Fe is on the y-axis** — if you ever swap those axes for a different plot, this line's geometry no longer represents the pyrite ratio and must be recomputed for the new axis orientation.


In [ ]:
# --- Pyrite tie-line (Escolme et al., 2017) ---
# Valid ONLY for S_pct on x-axis and Fe_pct on y-axis.
x0, y0 = 3, 2.5
x1, y1 = 6, 5

slope = (y1 - y0) / (x1 - x0)
intercept = y0 - slope * x0

x_range = [0, 15]
y_extrapolated = [slope * x + intercept for x in x_range]

diagram.add_shape(
    type='line',
    x0=x_range[0], y0=y_extrapolated[0],
    x1=x_range[1], y1=y_extrapolated[1],
    line=dict(color='red', width=1, dash='dash')
)

diagram.show()


## 8. Final layout tweaks

The last step just tidies the legend so it doesn't dominate the figure — smaller legend title/label font sizes are a nice touch once you have two legends stacked (the continuous As colorbar *and* the discrete As_Group symbol legend, as seen in the two example screenshots in this notebook's context, where the two legends visually overlap near the top-right — a good candidate for a follow-up fix, e.g. moving the legend via `legend=dict(y=...)` or hiding the colorbar title).


In [ ]:
diagram.update_layout(
    legend_title_font_size=10,
    legend_font_size=6
)

diagram.show()


## 9. Optional improvement: fixing the overlapping legends

In the two reference screenshots, the discrete `As_Group` symbol legend and the continuous `As_ppm` colorbar title overlap each other in the top-right corner. A quick fix is to explicitly reposition the discrete legend below the colorbar:


In [ ]:
diagram.update_layout(
    legend=dict(
        title_font_size=10,
        font_size=10,
        yanchor="top",
        y=0.85,          # nudge the discrete legend down...
        xanchor="left",
        x=1.05,          # ...and slightly right, clear of the colorbar
    ),
    coloraxis_colorbar=dict(
        title="As_ppm",
        y=0.5,
        len=0.6,
    ),
)

diagram.show()


## Summary

| Step | What happens | Why |
|---|---|---|
| 1 | Load 3 CSVs, concat, write `merged.csv` | Combine per-batch assays into one auditable table |
| 2 | Re-read `merged.csv`, select `Cu_ppm, Fe_pct, S_pct, As_ppm` | Decouple merge from clean; focus on the 4 relevant elements |
| 3 | `to_numeric(errors='coerce')` + `dropna` | Remove below-detection / non-numeric junk safely |
| 4 | `Cu_pct = Cu_ppm / 10000` | Put Cu on the same %-scale as Fe and S |
| 5 | Filter `Cu_pct > 1.0`; add `As_Group` | Focus on ore-grade Cu; flag high-As samples |
| 6 | `px.scatter(x=S_pct, y=Fe_pct, size=Cu_pct, color=As_ppm, symbol=As_Group)` | Encode 4 variables in one 2D chart |
| 7 | `add_shape` pyrite tie-line | Petrological Fe:S reference (Escolme et al., 2017) |
| 8 | `update_layout` font sizes | Legend readability |

To reuse this notebook on the real Mungana/Red Dome data, just replace the synthetic-data cell in **Section 1** with the original `pd.read_csv(...)` calls against `845.csv`, `883.csv`, and `997.csv` — every cell after that runs unchanged.
